# Visualize Rates
### Import libraries

In [ ]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm

In [ ]:
from utils import add_package_path

Paths

In [ ]:
# Current Working Directory
cwd = os.getcwd()
# WORKSPACE
ws = os.environ['WORKSPACE_PATH']

In [ ]:
prefix = '/home/zanardi/Workspace/ML/PyCOMET'
paths = {
    # PyCOMET source dir
    'pycomet': prefix + '/pycomet/',
    # SurQCT source dir
    'surqct': prefix + '/pycomet/examples/SurQCT/'
}
add_package_path(paths)

In [ ]:
# Output directory
out = cwd + '/Output/ExcitRates/Inel/final/c01.refine/'
if not os.path.exists(out):
    os.makedirs(out)

### Inputs

In [ ]:
SystemName   = 'O3_UMN'
RatesId      = 'Inel'
TTranVec     = [5000]
MinRateValue = 1.e-14
iIdxVec      = [200, 2550]

SuffixesHDF5 = {
    'QCT':    '/Orig/',
    'SurQCT': '/excit.qct_diss/c01.refine/'
}

### Initialize SurQCT objects

Import additional modules

In [ ]:
from surqct.src.chemical_systems import systems as syst
from surqct.src.rates_type.rates_type import RatesType
from surqct.src.utilities import utils
from surqct import config as cfg

In [ ]:
# Visualization libraries
matplotlib = utils.import_matplotlib()
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']
%matplotlib notebook

Initializing QCT & SurQCT chemical systems

In [ ]:
Systems = {}
for Identifier, SuffixHDF5 in SuffixesHDF5.items():
    # Initializing rates
    Systems[Identifier] = syst.get_system(
        NameLong=SystemName,
        KinSource=Identifier,
        ProcNames=tf.nest.flatten(RatesId),
        PathToHDF5Fld=cfg.paths.HDF5Dtb + '/' + Identifier + '/' + SuffixHDF5
    )
    # Reading rates
    Systems[Identifier].Processes.read_rates(TTranVec=TTranVec, MicroRev=True)

### Plotting rates

In [ ]:
def plot_prediction(
    Path,
    System,
    Data,
    iIdxVec,
    FigName=None,
    vmin=1.e-14,
    QuantumNumbers=False
):
    fig, ax = plt.subplots(2, 2, sharex=True, sharey=True)

    Mol = System.Molecules[0]
    if QuantumNumbers:
        x, y = Mol.DiatData['vqn'], Mol.DiatData['jqn']
    else:
        x, y = Mol.DiatData['EVib0'] - Mol.DissE, Mol.DiatData['ERot0']
    vmax = np.amax(tf.nest.flatten(Data))
    for i, iIdx in enumerate(iIdxVec):
        np.amax(tf.nest.flatten(Data))
        for j, (identifier, data) in enumerate(Data.items()):
            mappable = ax[i,j].scatter(
                x,
                y,
                c=data[str(iIdx)],
                cmap='jet',
                s=20,
                norm=matplotlib.colors.LogNorm(
                    vmin=vmin,
                    vmax=vmax
                )
            )

    fig.add_subplot(111, frame_on=False)
    plt.tick_params(labelcolor='none', which='both', top=False, bottom=False, left=False, right=False)
    plt.xticks([])
    plt.yticks([])
    # Labels
    if QuantumNumbers:
        labels = ['v', 'J']
    else:
        labels = ['$\epsilon_{Vib}$ [eV]', '$\epsilon_{Rot}$ [eV]']
    plt.xlabel(labels[0], labelpad=50)
    fig.text(0., 0.41, labels[1], va='center', ha='center', rotation='vertical', fontsize=plt.rcParams['axes.labelsize'])

    cbar = plt.colorbar(
        mappable,
        ax=ax.ravel().tolist(),
        orientation='horizontal',
        location='top',
        aspect=50
    )
    cbar.set_label(r'$k_{ij}^I$ [cm$^3$/s]', labelpad=30, size=30)
    cbar.ax.tick_params(labelsize=24)

    if FigName is None:
        FigName = 'diss3D'
    plt.savefig(Path + '/' + FigName + '.pdf')


In [ ]:
for i, TTran in enumerate(tqdm(TTranVec)):
    TTranStr = str(int(TTran))
    Data = {k: {} for k in Systems.keys()}
    for iIdx in iIdxVec:
        for Identifier, System in Systems.items():
            Data[Identifier][str(iIdx)] = System.Processes.Rates[RatesId][TTranStr][iIdx]
    plot_prediction(
        Path=out,
        System=Systems['QCT'],
        Data=Data,
        iIdxVec=iIdxVec,
        FigName=f'T{TTranStr}K',
        QuantumNumbers=True
    )